# Metadata

Domain: Material Science

Subdomain: Crystallography

References: https://doi.org/10.1093/biomet/45.1-2.229 (Mackenzie, Biometrika 45, 229-240, 1958); https://doi.org/10.1016/0001-6160(66)90168-4 (Brandon, Acta Metallurgica 14, 1479-1484, 1966); https://doi.org/10.1107/S056773947400043X (Grimmer, Bollmann and Warrington, Acta Cryst. A30, 197-207, 1974)


# Title

Random-Texture Statistics of Cubic Grain Boundaries: Exact Misorientation-Ball Overlaps, the Mackenzie Distribution and Brandon CSL Fractions


# Subproblem 1

## Prompt

In texture analysis and grain-boundary statistics, crystal orientations and misorientations are elements of the rotation group \\(SO(3)\\), and randomly oriented grains are described by the uniform (Haar) probability measure \\(\mu\\) on \\(SO(3)\\), normalized such that \\(\mu(SO(3))=1\\). A rotation is represented by a unit quaternion \\(q=(w,x,y,z)=(\cos\tfrac{\omega}{2},\thinspace\mathbf{n}\sin\tfrac{\omega}{2})\\), where \\(\omega\in[0^\circ,180^\circ]\\) is the rotation angle about the unit axis \\(\mathbf{n}\\); \\(q\\) and \\(-q\\) describe the same rotation. The distance \\(d(R_1,R_2)\in[0^\circ,180^\circ]\\) between two rotations is the rotation angle of \\(R_1R_2^{-1}\\). For a rotation \\(R\\) and a radius \\(r\in[0^\circ,180^\circ]\\), the ball of rotations within \\(r\\) of \\(R\\) is

\\(\displaystyle B(R,r)=\lbrace Q\in SO(3):\quad d(Q,R)\le r\rbrace.\\)

Such balls are used as tolerance regions around special misorientations. When several tolerance regions overlap, the probability that a random grain boundary satisfies two or three criteria at the same time is the Haar measure of the intersection of the corresponding balls.

Implement a function that returns

\\(\displaystyle P=\mu\Big(\bigcap_{i=1}^{k}B(R_i,r_i)\Big)\\)

for \\(k=1\\), \\(2\\) or \\(3\\) balls. The rotations are given by unit quaternions with arbitrary signs and the radii in degrees. For all radii in \\([0^\circ,180^\circ]\\), the returned value \\(\tilde P\\) must satisfy \\(|\tilde P-P|\le10^{-9}\thinspace P+10^{-13}\\). Use only NumPy and the Python standard library.

```python
import numpy as np

def ball_overlap_measure(quats, radii):
    """
    Haar measure of the intersection of one, two or three balls of rotations.

    Inputs:
        quats : array_like of float, shape (k, 4), k in {1, 2, 3}
            Unit quaternions (w, x, y, z) of the ball centres R_1, ..., R_k. q and -q denote the same rotation.
        radii : array_like of float, shape (k,)
            Radii r_1, ..., r_k of the balls in degrees, 0 <= r_i <= 180.

    Output:
        measure : float
            P = mu(B(R_1, r_1) ∩ ... ∩ B(R_k, r_k)), with mu the Haar probability measure on SO(3)
            (mu(SO(3)) = 1). Dimensionless; the returned value must satisfy |result - P| <= 1e-9 * P + 1e-13.
    """
    ...
```


## Background

**Haar measure.** The Haar measure \\(\mu\\) is invariant under multiplication of all rotations from the left or from the right by a fixed rotation. The rotation angle \\(\omega\\) of a random rotation has the probability density \\((1-\cos\omega)/\pi\\) on \\([0,\pi]\\), so a single ball has the measure \\(\mu(B(R,r))=(r-\sin r)/\pi\\) (with \\(r\\) in radians). Because of the invariance, the measure of an intersection of balls depends only on the radii and on the relative positions of the centres.

**Quaternions.** Unit quaternions form the unit sphere \\(S^3\subset\mathbb{R}^4\\), and the map \\(q\mapsto R\\) is two-to-one onto \\(SO(3)\\). The Haar measure on \\(SO(3)\\) corresponds to the uniform measure on \\(S^3\\). The distance between two rotations satisfies \\(\cos\big(d(R_1,R_2)/2\big)=|q_1\cdot q_2|\\), where the absolute value accounts for the two quaternions \\(\pm q\\) of each rotation.

**Special cases.** Balls whose centres are more than \\(r_1+r_2\\) apart do not intersect, a ball of radius \\(180^\circ\\) is the whole group, \\(B(R_2,r_2)\subseteq B(R_1,r_1)\\) whenever \\(d(R_1,R_2)+r_2\le r_1\\), and three balls can overlap pairwise without having a common point.

References: J. K. Mackenzie, Biometrika 45, 229-240 (1958), https://doi.org/10.1093/biomet/45.1-2.229


## Testing Template

In [ ]:
import math
import numpy as np


def _quat(axis, angle_deg):
    n = np.asarray(axis, dtype=float)
    n = n / np.linalg.norm(n)
    t = math.radians(angle_deg) / 2.0
    return np.array([math.cos(t), *(math.sin(t) * n)])


def _qmul(a, b):
    w1, x1, y1, z1 = a
    w2, x2, y2, z2 = b
    return np.array([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2, w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2, w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2])


def _ball(r_deg):
    r = math.radians(r_deg)
    return (r - math.sin(r)) / math.pi


def _close(value, exact):
    return abs(value - exact) <= 1e-9 * exact + 1e-13


BASE = _quat([2, -1, 1], 71.0)


def _rel(axis, omega):
    # rotation at distance omega from BASE
    return _qmul(BASE, _quat(axis, omega))


def test_case_1():
    # a single ball
    for r in [0.0, 1.0, 15.0, 60.0, 120.0, 180.0]:
        assert _close(ball_overlap_measure([BASE], [r]), _ball(r))
    # two balls: identical centres, nested, disjoint, touching, whole group
    q2 = _rel([1, -1, 2], 20.0)
    q3 = _rel([0, 1, 1], 60.0)
    assert _close(ball_overlap_measure([BASE, -BASE], [20.0, 70.0]), _ball(20.0))
    assert _close(ball_overlap_measure([BASE, q2], [30.0, 5.0]), _ball(5.0))
    assert _close(ball_overlap_measure([-q2, BASE], [5.0, 25.0]), _ball(5.0))
    assert _close(ball_overlap_measure([BASE, q3], [20.0, 30.0]), 0.0)
    assert _close(ball_overlap_measure([BASE, -q3], [25.0, 35.0]), 0.0)
    assert _close(ball_overlap_measure([BASE, q3], [180.0, 35.0]), _ball(35.0))
    # three balls that reduce to two: a repeated ball, a third ball of radius 180, a third ball containing the others
    q4 = _rel([1, 2, 3], 40.0)
    assert _close(ball_overlap_measure([BASE, BASE, q4], [30.0, 30.0, 25.0]), 0.0005914821840388177)
    assert _close(ball_overlap_measure([BASE, q4, _rel([3, 0, 1], 77.0)], [30.0, 25.0, 180.0]), 0.0005914821840388177)
    assert _close(ball_overlap_measure([BASE, _rel([1, 0, 0], 20.0), _rel([1, 0, 0], 10.0)], [30.0, 25.0, 60.0]),
                  0.0027127581021668865)
    # three balls that overlap pairwise but have no common point
    triple = [BASE, _rel([1, 0, 0], 40.0), _rel([0, 1, 0], 40.0)]
    assert _close(ball_overlap_measure(triple[:2], [22.0, 22.0]), 3.644710898684042e-05)
    assert _close(ball_overlap_measure(triple, [22.0, 22.0, 22.0]), 0.0)


def test_case_2():
    # three partially overlapping balls
    cases = [
        ([BASE, _rel([1, 0, 0], 20.0), _rel([0, 1, 0], 25.0)], [30.0, 25.0, 35.0], 0.0016764681914635668),
        ([BASE, _rel([1, 2, 3], 40.0), _rel([3, -1, 1], 35.0)], [45.0, 40.0, 30.0], 0.001881204843550621),
        ([BASE, _rel([1, 1, 0], 12.0), _rel([1, -1, 2], 15.0)], [10.0, 8.0, 12.0], 4.356606272218225e-07),
        ([BASE, _rel([0, 0, 1], 60.0), _rel([0, 0, 1], 30.0)], [40.0, 35.0, 20.0], 0.0007907444062888634),
        ([BASE, _rel([2, 1, -1], 50.0), _rel([-1, 3, 2], 55.0)], [50.0, 45.0, 40.0], 0.00027656198542667075),
    ]
    for quats, radii, exact in cases:
        assert _close(ball_overlap_measure(quats, radii), exact)
        # order of the balls and signs of the quaternions do not matter
        assert _close(ball_overlap_measure([quats[2], -quats[0], quats[1]], [radii[2], radii[0], radii[1]]), exact)
    # Brandon tolerance balls of the Sigma 13, Sigma 25 and Sigma 37 rotations about [100] (theta0 = 15 degrees)
    q13 = _qmul(BASE, _quat([1, 0, 0], 2.0 * math.degrees(math.atan(1.0 / 5.0))))
    q25 = _qmul(BASE, _quat([1, 0, 0], 2.0 * math.degrees(math.atan(1.0 / 7.0))))
    q37 = _qmul(BASE, _quat([1, 0, 0], 2.0 * math.degrees(math.atan(1.0 / 6.0))))
    radii = [15.0 / math.sqrt(13.0), 15.0 / math.sqrt(25.0), 15.0 / math.sqrt(37.0)]
    assert _close(ball_overlap_measure([q13, q25, q37], radii), 2.265370018578308e-07)
    assert _close(ball_overlap_measure([q13, q37], [radii[0], radii[2]]), 2.2136551013675917e-06)
    # invariance under a common rotation from the left
    g = _quat([3, 0, -1], 123.0)
    quats = [_qmul(g, q) for q in cases[0][0]]
    assert _close(ball_overlap_measure(quats, cases[0][1]), cases[0][2])


def test_case_3():
    # large balls: several combinations of the quaternion signs contribute
    cases = [
        ([BASE, _rel([1, 2, 3], 150.0), _rel([3, -2, 1], 120.0)], [120.0, 110.0, 100.0], 0.017200278703601368),
        ([BASE, _rel([1, 0, 1], 170.0), _rel([0, 1, 1], 160.0)], [150.0, 140.0, 130.0], 0.15441953059530852),
        ([BASE, _rel([2, 1, 2], 100.0), _rel([1, -2, 0], 179.0)], [170.0, 160.0, 175.0], 0.6526039320023092),
        ([BASE, _rel([1, 1, 1], 180.0), _rel([1, -1, 0], 90.0)], [100.0, 100.0, 100.0], 0.0011116742393344912),
        ([BASE, _rel([1, 2, 3], 170.0)], [100.0, 100.0], 0.012244441820474994),
        ([BASE, _rel([1, 2, 3], 120.0)], [170.0, 150.0], 0.59651327103954438),
        ([BASE, _rel([1, 2, 3], 180.0)], [120.0, 120.0], 0.10031432409533196),
    ]
    for quats, radii, exact in cases:
        assert _close(ball_overlap_measure(quats, radii), exact)
        assert _close(ball_overlap_measure([-q for q in quats[::-1]], radii[::-1]), exact)


## Solution

In [ ]:
import math

import numpy as np


def ball_overlap_measure(quats, radii):
    quats = np.atleast_2d(np.asarray(quats, dtype=float))
    quats = quats / np.linalg.norm(quats, axis=1)[:, None]
    half = 0.5 * np.radians(np.asarray(radii, dtype=float).reshape(-1))   # cap radii on the quaternion sphere S^3
    k = len(half)
    if np.any(half <= 0.0):
        return 0.0
    # integrate over the geodesic distance psi from the centre of the smallest cap
    order = np.argsort(half)
    quats, half = quats[order], half[order]
    p, alpha = quats[0], half[0]
    rules = []
    for n in (20, 40):
        nodes, weights = np.polynomial.legendre.leggauss(n)
        s_map = 0.5 * (nodes + 1.0)
        # smoothstep map: removes square-root behaviour at panel ends
        rules.append((s_map * s_map * (3.0 - 2.0 * s_map), 3.0 * s_map * (1.0 - s_map) * weights))

    def slice_geometry(q, beta, psi):
        # the slice {x on S^3 : angle(x, p) = psi} is a 2-sphere; its part inside cap(q, beta) is a spherical cap
        # (centred on the direction of q seen from p) with angular radius a(psi); returns cos(a) (a = 0: empty, pi: full)
        # and the angle d between p and q
        c = float(p @ q)
        s = float(np.linalg.norm(q - c * p))
        d = math.atan2(s, c)                                  # angle between p and q
        sp, cp = np.sin(psi), np.cos(psi)
        full = (psi + d <= beta) | (psi + d >= 2.0 * math.pi - beta)
        empty = np.abs(psi - d) >= beta
        with np.errstate(divide="ignore", invalid="ignore"):
            t = (math.cos(beta) - cp * math.cos(d)) / (sp * math.sin(d))
        t = np.where(full, -1.0, np.where(empty, 1.0, np.clip(t, -1.0, 1.0)))
        return t, d

    def lens_fraction(ca, cb, g):
        # fraction of the unit 2-sphere inside two caps (cosines of the radii ca, cb; centre separation g)
        a, b = np.arccos(ca), np.arccos(cb)
        sa, sb = np.sin(a), np.sin(b)
        out = np.zeros_like(a)
        small = np.minimum(a, b)
        contain = g <= np.abs(a - b)
        cover = a + b >= 2.0 * math.pi - g
        part = (g < a + b) & ~contain & ~cover & (sa > 0) & (sb > 0)
        out = np.where(contain, 0.5 * (1.0 - np.cos(small)), out)
        out = np.where(cover & ~contain, 0.5 * (1.0 - ca) + 0.5 * (1.0 - cb) - 1.0, out)
        if np.any(part) and math.sin(g) > 0.0:
            with np.errstate(divide="ignore", invalid="ignore"):
                x = np.clip((math.cos(g) - ca * cb) / (sa * sb), -1.0, 1.0)
                y = np.clip((cb - math.cos(g) * ca) / (math.sin(g) * sa), -1.0, 1.0)
                z = np.clip((ca - math.cos(g) * cb) / (math.sin(g) * sb), -1.0, 1.0)
                lens = 2.0 * (math.pi - np.arccos(x) - ca * np.arccos(y) - cb * np.arccos(z)) / (4.0 * math.pi)
            out = np.where(part, lens, out)
        out = np.where((a <= 0.0) | (b <= 0.0), 0.0, out)
        return out

    def area_fraction(others, psi):
        # fraction of the slice 2-sphere at distance psi from p that lies inside all other caps
        if not others:
            return np.ones_like(psi)
        if len(others) == 1:
            q, beta = others[0]
            t, _ = slice_geometry(q, beta, psi)
            return 0.5 * (1.0 - t)
        (q2, b2), (q3, b3) = others
        t2, _ = slice_geometry(q2, b2, psi)
        t3, _ = slice_geometry(q3, b3, psi)
        # angle between the directions of q2 and q3 as seen from p (tangent directions at p)
        u2 = q2 - float(p @ q2) * p
        u3 = q3 - float(p @ q3) * p
        n2, n3 = np.linalg.norm(u2), np.linalg.norm(u3)
        if n2 == 0.0 or n3 == 0.0:                            # a cap centred on +-p: it is either full or empty on the slice
            return np.where(t2 <= -1.0, 0.5 * (1.0 - t3), 0.0) if n2 == 0.0 else np.where(t3 <= -1.0, 0.5 * (1.0 - t2), 0.0)
        g = math.atan2(float(np.linalg.norm(u2 / n2 - u3 / n3)), float(np.linalg.norm(u2 / n2 + u3 / n3))) * 2.0
        return lens_fraction(t2, t3, g)

    def cap_intersection(others):
        # fraction of S^3 inside cap(p, alpha) and all other caps: (2/pi) int_0^alpha sin(psi)^2 A(psi) dpsi
        cuts = {0.0, alpha}
        for q, beta in others:
            d = math.atan2(float(np.linalg.norm(q - float(p @ q) * p)), float(p @ q))
            cuts.update(x for x in (beta - d, d - beta, d + beta, 2.0 * math.pi - beta - d) if 0.0 < x < alpha)
        if len(others) == 2:
            # regime changes of the two slice caps (touching, containment, covering) located by bisection
            grid = np.linspace(0.0, alpha, 801)[1:-1]
            (q2, b2), (q3, b3) = others
            u2 = q2 - float(p @ q2) * p
            u3 = q3 - float(p @ q3) * p
            if np.linalg.norm(u2) > 0.0 and np.linalg.norm(u3) > 0.0:
                g = 2.0 * math.atan2(float(np.linalg.norm(u2 / np.linalg.norm(u2) - u3 / np.linalg.norm(u3))),
                                     float(np.linalg.norm(u2 / np.linalg.norm(u2) + u3 / np.linalg.norm(u3))))

                def events(x):
                    s2, _ = slice_geometry(q2, b2, np.atleast_1d(x))
                    s3, _ = slice_geometry(q3, b3, np.atleast_1d(x))
                    a, b = np.arccos(s2), np.arccos(s3)
                    return np.stack([a + b - g, a - b - g, b - a - g, a + b - (2.0 * math.pi - g)], axis=-1)

                vals = events(grid)
                lo_list, hi_list, col_list = [], [], []
                for m in range(4):
                    sign = np.sign(vals[:, m])
                    for i in np.nonzero(sign[:-1] * sign[1:] < 0)[0]:
                        lo_list.append(grid[i])
                        hi_list.append(grid[i + 1])
                        col_list.append(m)
                if lo_list:
                    lo, hi, col = np.array(lo_list), np.array(hi_list), np.array(col_list)
                    rows = np.arange(len(col))
                    flo = events(lo)[rows, col]
                    for _ in range(60):                   # bisection of all brackets at once
                        mid = 0.5 * (lo + hi)
                        fm = events(mid)[rows, col]
                        same = (fm < 0) == (flo < 0)
                        lo = np.where(same, mid, lo)
                        flo = np.where(same, fm, flo)
                        hi = np.where(same, hi, mid)
                    cuts.update(float(x) for x in 0.5 * (lo + hi))
        cuts = sorted(cuts)

        def integrand(psi, lo, hi):
            if len(others) == 1:
                q, beta = others[0]
                d = math.atan2(float(np.linalg.norm(q - float(p @ q) * p)), float(p @ q))
                mid = 0.5 * (lo + hi)
                if abs(mid - d) >= beta:
                    return np.zeros_like(psi)
                if mid + d <= beta or mid + d >= 2.0 * math.pi - beta:
                    return np.sin(psi) ** 2
                # cancellation-free form of sin(psi)^2 (1 - t) / 2
                return np.sin(psi) * np.sin(0.5 * (beta + psi - d)) * np.sin(0.5 * (beta - psi + d)) / math.sin(d)
            return np.sin(psi) ** 2 * area_fraction(others, psi)

        def panel(lo, hi, depth):
            coarse, fine = [(hi - lo) * float(np.sum(w * integrand(lo + (hi - lo) * h, lo, hi))) for h, w in rules]
            if abs(fine - coarse) <= 1e-16 + 1e-12 * abs(fine) or depth >= 40:
                return fine
            mid = 0.5 * (lo + hi)
            return panel(lo, mid, depth + 1) + panel(mid, hi, depth + 1)

        total = 0.0
        for lo, hi in zip(cuts[:-1], cuts[1:]):
            if hi > lo:
                total += panel(lo, hi, 0)
        return 2.0 / math.pi * total

    # each ball corresponds to the two antipodal caps around +q and -q; the caps around +p and -p give equal shares
    total = 0.0
    signs = [()] if k == 1 else ([(1.0,), (-1.0,)] if k == 2 else [(1.0, 1.0), (1.0, -1.0), (-1.0, 1.0), (-1.0, -1.0)])
    for sg in signs:
        others = [(s * quats[j + 1], half[j + 1]) for j, s in enumerate(sg)]
        total += cap_intersection(others)
    return 2.0 * total


# Subproblem 2

## Prompt

For a crystal with cubic symmetry, the 24 proper rotations that map a cube onto itself form the group \\(G\\): the identity, the rotations by \\(\pm90^\circ\\) and \\(180^\circ\\) about the three cube axes \\(\langle100\rangle\\), the rotations by \\(\pm120^\circ\\) about the four body diagonals \\(\langle111\rangle\\), and the rotations by \\(180^\circ\\) about the six face diagonals \\(\langle110\rangle\\). The misorientation \\(M\\) between two cubic grains has the crystallographically equivalent descriptions \\(O_1MO_2\\) with \\(O_1,O_2\in G\\), and its disorientation angle is the smallest rotation angle among them,

\\(\displaystyle \theta_{\mathrm{dis}}(M)=\min_{O_1,O_2\in G}\quad\omega(O_1MO_2),\\)

where \\(\omega(\cdot)\in[0^\circ,180^\circ]\\) denotes the rotation angle.

For grains with uncorrelated orientations (random texture), \\(M\\) is distributed according to the uniform (Haar) probability measure \\(\mu\\) on \\(SO(3)\\), normalized such that \\(\mu(SO(3))=1\\). Implement a function that returns the cumulative distribution function of the disorientation angle,

\\(\displaystyle F(\theta)=\mu\big(\lbrace M\in SO(3):\quad\theta_{\mathrm{dis}}(M)\le\theta\rbrace\big),\\)

for a given \\(\theta\\) in degrees, \\(0\le\theta\le180\\). The result must have an absolute error below \\(10^{-10}\\) for every \\(\theta\\) in this range. Use only NumPy and the Python standard library.

```python
import numpy as np

def cubic_disorientation_cdf(theta):
    """
    Cumulative distribution function of the disorientation angle of randomly misoriented cubic grains.

    Input:
        theta : float
            Disorientation angle in degrees, 0 <= theta <= 180.

    Output:
        F : float
            Probability that a Haar-random misorientation M has theta_dis(M) <= theta, where theta_dis is the
            smallest rotation angle among the equivalent descriptions O1 M O2 (O1, O2 in the 24-element proper
            rotation group of the cube). Dimensionless; absolute error below 1e-10.
    """
    ...
```


## Background

**The Mackenzie distribution.** The distribution of the disorientation angle for randomly oriented cubic grains was derived by Mackenzie (1958). It is the reference against which measured misorientation distributions (for example from EBSD maps) are compared to detect texture or special grain-boundary populations, and it differs strongly from the distribution of the rotation angle itself because the 24 symmetry-equivalent descriptions allow the smallest angle to be chosen.

**Haar measure and rotation angles.** The Haar measure is invariant under multiplication from the left and from the right by fixed rotations, and the rotation angle is invariant under conjugation, \\(\omega(QMQ^{-1})=\omega(M)\\). The rotation angle of a Haar-random rotation has the probability density \\((1-\cos\omega)/\pi\\) on \\([0,\pi]\\), so the set of rotations with rotation angle at most \\(r\\) has the measure \\((r-\sin r)/\pi\\) (with \\(r\\) in radians). A rotation by \\(\omega\\) about the unit axis \\(\mathbf{n}\\) corresponds to the unit quaternions \\(\pm(\cos\tfrac{\omega}{2},\mathbf{n}\sin\tfrac{\omega}{2})\\), and the Haar measure corresponds to the uniform measure on the unit sphere \\(S^3\\) of quaternions. Equivalently, a rotation can be described by its Rodrigues vector \\(\mathbf{n}\tan\tfrac{\omega}{2}\\).

**Limits.** \\(F(0)=0\\), \\(F\\) is non-decreasing, and \\(F(\theta)=1\\) for all angles at or above the largest possible disorientation angle of cubic grains.

References: J. K. Mackenzie, Biometrika 45, 229-240 (1958), https://doi.org/10.1093/biomet/45.1-2.229


## Testing Template

In [ ]:
import math


def test_case_1():
    # small angles: the rotations within theta of different symmetry elements do not overlap
    for theta in [0.0, 0.5, 5.0, 15.0, 30.0, 40.0, 45.0]:
        t = math.radians(theta)
        assert abs(cubic_disorientation_cdf(theta) - 24.0 * (t - math.sin(t)) / math.pi) < 1e-10
    # monotonic and bounded
    values = [cubic_disorientation_cdf(0.5 * k) for k in range(0, 361, 7)]
    assert all(b >= a - 1e-12 for a, b in zip(values, values[1:]))
    assert all(-1e-12 <= v <= 1.0 + 1e-12 for v in values)


def test_case_2():
    # intermediate angles
    cases = [
        (46.0, 0.6365159050652123),
        (48.5, 0.7264893856490133),
        (50.0, 0.7758566679457417),
        (53.0, 0.8627673510980809),
        (56.5, 0.9415150953189559),
        (59.0, 0.9808473648811813),
        (60.0, 0.992280125488668),
        (60.5, 0.9963573413402229),
        (60.72, 0.9975130325490773),
    ]
    for theta, expected in cases:
        assert abs(cubic_disorientation_cdf(theta) - expected) < 1e-10


def test_case_3():
    # angles close to the largest disorientation angle
    cases = [
        (60.8, 0.9978393994226368),
        (61.2, 0.9989880423818288),
        (61.7, 0.9996972916277741),
        (62.2, 0.9999540616336734),
        (62.6, 0.9999983827022721),
        (62.78, 0.9999999985319115),
        (62.795, 0.9999999999826239),
    ]
    for theta, expected in cases:
        assert abs(cubic_disorientation_cdf(theta) - expected) < 1e-10
    for theta in [62.8, 63.0, 75.0, 90.0, 135.0, 180.0]:
        assert abs(cubic_disorientation_cdf(theta) - 1.0) < 1e-10


## Solution

In [ ]:
import math

import numpy as np


def cubic_disorientation_cdf(theta):
    theta = float(theta)
    if theta <= 0.0:
        return 0.0
    if theta >= 180.0:
        return 1.0

    # Fundamental zone of the 24 proper cubic rotations in Rodrigues space r = n tan(omega/2): a truncated cube bounded by
    # |r_i| <= tan(pi/8) (from the 90 degree rotations) and |r_1| + |r_2| + |r_3| <= 1 (from the 120 degree rotations).
    t8 = math.tan(math.pi / 8.0)
    normals, dists = [], []
    for k in range(3):
        for sgn in (1.0, -1.0):
            n = np.zeros(3)
            n[k] = sgn
            normals.append(n)
            dists.append(t8)
    for sx in (1.0, -1.0):
        for sy in (1.0, -1.0):
            for sz in (1.0, -1.0):
                normals.append(np.array([sx, sy, sz]) / math.sqrt(3.0))
                dists.append(1.0 / math.sqrt(3.0))
    normals = np.array(normals)
    dists = np.array(dists)
    cos_pairs = np.clip(normals @ normals.T, -1.0, 1.0)
    rho_edge = math.sqrt(2.0) * t8                              # nearest points of the edges
    rho_vertex = math.sqrt(2.0 * t8 * t8 + (1.0 - 2.0 * t8) ** 2)  # vertices (disorientation 62.80 degrees)

    def inside_fraction(rho):
        # fraction of the sphere |r| = rho inside the zone: caps beyond each face, corrected by the pairwise overlaps of
        # caps (edges); for rho below the vertex distance no point of the sphere lies beyond three faces
        if rho >= rho_vertex:
            return 0.0
        cap = np.arccos(np.clip(dists / rho, -1.0, 1.0))        # angular radii of the caps (0 if the face is not reached)
        f = 1.0 - np.sum(0.5 * (1.0 - np.cos(cap)))
        m = len(dists)
        for i in range(m):
            for j in range(i + 1, m):
                a, b = cap[i], cap[j]
                if a <= 0.0 or b <= 0.0:
                    continue
                c = math.acos(cos_pairs[i, j])
                if c >= a + b:
                    continue
                if c <= abs(a - b):
                    lens = 2.0 * math.pi * (1.0 - math.cos(min(a, b)))
                else:
                    x = (math.cos(c) - math.cos(a) * math.cos(b)) / (math.sin(a) * math.sin(b))
                    y = (math.cos(b) - math.cos(c) * math.cos(a)) / (math.sin(c) * math.sin(a))
                    z = (math.cos(a) - math.cos(c) * math.cos(b)) / (math.sin(c) * math.sin(b))
                    lens = 2.0 * (math.pi - math.acos(max(-1.0, min(1.0, x)))
                                  - math.cos(a) * math.acos(max(-1.0, min(1.0, y)))
                                  - math.cos(b) * math.acos(max(-1.0, min(1.0, z))))
                f += lens / (4.0 * math.pi)
        return f

    # Haar measure in Rodrigues space: d(mu) = dV / (pi^2 (1 + rho^2)^2); rotation angle <= theta  <=>  rho <= tan(theta/2)
    rho_max = min(math.tan(math.radians(theta) / 2.0) if theta < 180.0 else math.inf, rho_vertex)
    breaks = [0.0] + [x for x in (t8, 1.0 / math.sqrt(3.0), rho_edge) if x < rho_max] + [rho_max]
    nodes, weights = np.polynomial.legendre.leggauss(60)
    s = 0.5 * (nodes + 1.0)
    h = s * s * (3.0 - 2.0 * s)                                 # smoothstep map: removes endpoint singularities
    dh = 6.0 * s * (1.0 - s)
    total = 0.0
    for lo, hi in zip(breaks[:-1], breaks[1:]):
        rho = lo + (hi - lo) * h
        vals = np.array([inside_fraction(x) for x in rho]) * rho ** 2 / (1.0 + rho ** 2) ** 2
        total += 0.5 * (hi - lo) * float(np.sum(weights * vals * dh))
    return min(1.0, 24.0 * 4.0 / math.pi * total)


# Main Problem

## Prompt

Grain-boundary engineering programmes report the fraction of coincidence-site-lattice (CSL) boundaries of each index \\(\Sigma\\) measured by EBSD, and these fractions are only meaningful in comparison with the fractions expected for a random (untextured) cubic polycrystal. Compute these random-texture reference fractions exactly for Brandon's criterion. All conventions are fixed below.

**Rotations.** Rotations are elements of \\(SO(3)\\), expressed in the cubic crystal axes. The distance \\(d(R_1,R_2)\in[0^\circ,180^\circ]\\) between two rotations is the rotation angle of \\(R_1R_2^{-1}\\), and \\(B(R,r)=\lbrace Q\in SO(3):\quad d(Q,R)\le r\rbrace \\) is the ball of radius \\(r\\) around \\(R\\). The misorientation \\(M\\) of a random boundary is distributed according to the uniform (Haar) probability measure \\(\mu\\) on \\(SO(3)\\), normalized such that \\(\mu(SO(3))=1\\).

**\\(\Sigma\\) rotations.** A proper rotation matrix \\(R\\) (\\(R^{\mathsf T}R=I\\), \\(\det R=+1\\)) is a \\(\Sigma\\) rotation if \\(\Sigma\\) is the smallest positive integer for which all nine entries of \\(\Sigma R\\) are integers. Let \\(S_\Sigma\\) be the set of all \\(\Sigma\\) rotations. \\(S_1\\) is the set of the 24 proper rotations of the cube.

**Brandon's criterion.** With the Brandon constant \\(\theta_0\\), the misorientation \\(M\\) satisfies the criterion for the odd index \\(\Sigma\\) if \\(M\in U_\Sigma=\bigcup_{R\in S_\Sigma}B(R,\theta_0/\sqrt{\Sigma})\\), that is, if \\(M\\) lies within \\(\theta_0/\sqrt{\Sigma}\\) of at least one \\(\Sigma\\) rotation. For \\(\Sigma=1\\) this means that the disorientation angle of the boundary is at most \\(\theta_0\\) (low-angle boundary). A boundary is classified as the smallest odd \\(\Sigma\le\Sigma_{\max}\\) whose criterion it satisfies, so that the set of boundaries classified as \\(\Sigma\\) is \\(U_\Sigma\setminus\bigcup_{\Sigma'<\Sigma}U_{\Sigma'}\\) (union over odd \\(\Sigma'\\)).

Implement the function below. It returns a NumPy array \\(F\\) with one entry for each odd \\(\Sigma=1,3,5,\dots\le\Sigma_{\max}\\), in increasing order of \\(\Sigma\\), where \\(F_k=\mu\big(U_\Sigma\setminus\bigcup_{\Sigma'<\Sigma}U_{\Sigma'}\big)\\) for \\(\Sigma=2k+1\\) is the probability that a random boundary is classified as \\(\Sigma\\). The arguments satisfy the following condition: no rotation lies in four or more of the balls \\(B(R,\theta_0/\sqrt{\Sigma})\\) with \\(R\in S_\Sigma\\) and odd \\(\Sigma\le\Sigma_{\max}\\) (each of the 24 balls around the elements of \\(S_1\\) counted separately). Every returned entry \\(\tilde F_k\\) must satisfy \\(|\tilde F_k-F_k|\le10^{-7}\thinspace F_k+10^{-13}\\). Use only NumPy and the Python standard library.

```python
import numpy as np

def random_csl_fractions(sigma_max, theta0=15.0):
    """
    Fractions of random (Haar-distributed) cubic grain boundaries classified as Sigma by Brandon's criterion.

    Inputs:
        sigma_max : int
            Largest CSL index considered (>= 1). Only odd Sigma = 1, 3, 5, ... <= sigma_max are used.
        theta0 : float, default 15.0
            Brandon constant in degrees; the tolerance for index Sigma is theta0 / sqrt(Sigma) degrees.

    Output:
        fractions : numpy.ndarray of float, shape ((sigma_max + 1) // 2,)
            fractions[k] is the probability that a random boundary is classified as Sigma = 2k + 1, i.e. that its
            misorientation lies within theta0 / sqrt(Sigma) of a Sigma rotation but not within theta0 / sqrt(Sigma')
            of any Sigma' rotation with odd Sigma' < Sigma. Every entry satisfies |result - exact| <= 1e-7 * exact + 1e-13.
    """
    ...
```


## Background

**CSL boundaries and Brandon's criterion.** Two cubic lattices related by a rotation \\(R\\) share a coincidence-site lattice only if \\(R\\) has rational entries in the cubic axes, and the index \\(\Sigma\\) is the smallest positive integer such that \\(\Sigma R\\) is an integer matrix; only odd indices occur. Brandon proposed that a boundary belongs to \\(\Sigma\\) if its misorientation deviates from an exact \\(\Sigma\\) rotation by at most \\(15^\circ/\sqrt{\Sigma}\\). The \\(\Sigma=1\\) balls are centred on the 24 symmetry rotations of the cube, so \\(U_1\\) is the set of misorientations whose disorientation angle is at most \\(\theta_0\\). Every set \\(U_\Sigma\\) is invariant under \\(M\to O_1MO_2\\) with \\(O_1,O_2\in S_1\\), so the classification does not depend on the choice among equivalent descriptions of a boundary. Since the tolerance balls of different \\(\Sigma\\) rotations can overlap, the smallest-index rule assigns every boundary that satisfies at least one of the criteria for odd \\(\Sigma\le\Sigma_{\max}\\) to exactly one class, and the fraction of a class is smaller than the total measure of its balls when they overlap balls of smaller index or each other. Boundaries that satisfy none of these criteria are general (random) boundaries and belong to no class, so the fractions of all classes add up to the measure of \\(\bigcup_{\Sigma}U_\Sigma\\), which is less than 1.

**Measures.** The Haar measure is invariant under multiplication from the left and from the right by fixed rotations. The rotation angle \\(\omega\\) of a random rotation has the probability density \\((1-\cos\omega)/\pi\\) on \\([0,\pi]\\), so \\(\mu(B(R,r))=(r-\sin r)/\pi\\) (with \\(r\\) in radians), and the measure of the intersection of two balls depends only on their radii and on the distance between their centres. Rotations correspond two-to-one to unit quaternions \\(\pm q\\), the Haar measure to the uniform measure on the unit sphere \\(S^3\\), and \\(\cos\big(d(R_1,R_2)/2\big)=|q_1\cdot q_2|\\).

**\\(\Sigma\\) rotations.** \\(S_\Sigma\\) is empty for even \\(\Sigma\\), is closed under multiplication from either side by the 24 elements of \\(S_1\\), and contains \\(24\thinspace\psi(\Sigma)\\) rotations with \\(\psi(\Sigma)=\Sigma\prod_{p\mid\Sigma}(1+1/p)\\) (product over the distinct prime divisors). For example, \\(S_3\\) contains the \\(60^\circ\\) rotation about \\([111]\\), and \\(S_5\\) the rotation by \\(2\arctan(1/3)\approx36.87^\circ\\) about \\([100]\\).

**Example.** For \\(\theta_0=15^\circ\\) the \\(\Sigma3\\) balls (radius \\(8.66^\circ\\)) overlap neither each other nor the low-angle balls, so the random \\(\Sigma3\\) fraction is \\(96\thinspace(r-\sin r)/\pi\approx1.76\\)% with \\(r=15^\circ/\sqrt3\\).

References: D. G. Brandon, Acta Metallurgica 14, 1479-1484 (1966), https://doi.org/10.1016/0001-6160(66)90168-4; J. K. Mackenzie, Biometrika 45, 229-240 (1958), https://doi.org/10.1093/biomet/45.1-2.229; H. Grimmer, W. Bollmann and D. H. Warrington, Acta Cryst. A30, 197-207 (1974), https://doi.org/10.1107/S056773947400043X


## Testing Template

In [ ]:
import math
import numpy as np


def _ball(r_deg):
    r = math.radians(r_deg)
    return (r - math.sin(r)) / math.pi


def _psi(n):
    result, m, p = n, n, 2
    while p * p <= m:
        if m % p == 0:
            result = result // p * (p + 1)
            while m % p == 0:
                m //= p
        p += 1
    if m > 1:
        result = result // m * (m + 1)
    return result


def test_case_1():
    # tolerance balls that do not overlap: every class receives 24 psi(Sigma) full balls
    for sigma_max, theta0 in [(1, 15.0), (3, 15.0), (9, 15.0), (7, 18.0)]:
        sigmas = list(range(1, sigma_max + 1, 2))
        expected = np.array([24 * _psi(s) * _ball(theta0 / math.sqrt(s)) for s in sigmas])
        result = random_csl_fractions(sigma_max, theta0)
        assert result.shape == (len(sigmas),)
        assert np.allclose(result, expected, rtol=1e-7, atol=1e-13)
    # an even sigma_max adds no class
    result = random_csl_fractions(10, 15.0)
    assert result.shape == (5,)
    assert np.allclose(result, random_csl_fractions(9, 15.0), rtol=1e-7, atol=1e-13)


def test_case_2():
    # overlaps of at most two balls (Sigma 5 balls overlap each other, larger Sigma balls overlap smaller-index balls)
    cases = [
        (5, 20.0, [0.05382523719728907, 0.04160335142019564, 0.028851408770627508]),
        (5, 22.0, [0.07154976259933674, 0.05535044819155914, 0.03779935765230024]),
        (15, 22.0, [0.07154976259933674, 0.05535044819155914, 0.03779935765230024, 0.031102537421473085,
                    0.03200899332193516, 0.02251205359981105, 0.015676161399038373, 0.029762621458041625]),
        (19, 18.0, [0.039284056599681434, 0.030340548812247493, 0.021165441840921694, 0.017041055591549174,
                    0.01753634524032366, 0.012979474826330705, 0.01155813345813377, 0.016303889062140843,
                    0.008001364479977987, 0.008816361458549431]),
    ]
    for sigma_max, theta0, expected in cases:
        result = random_csl_fractions(sigma_max, theta0)
        assert result.shape == (len(expected),)
        assert np.allclose(result, expected, rtol=1e-7, atol=1e-13)
    # without overlaps the Sigma 5 class would receive 144 full balls
    assert random_csl_fractions(5, 20.0)[2] < 144 * _ball(20.0 / math.sqrt(5)) - 1e-6


def test_case_3():
    # three balls share points: low-angle, Sigma 5, Sigma 7, Sigma 13, Sigma 17 and Sigma 19 balls about common axes
    cases = [
        (13, 28.0, [0.14683478012765502, 0.11393722136643043, 0.06969980940595834, 0.05810699060760526,
                    0.06595649469939538, 0.03317930111281628, 0.003966413903656242]),
        (15, 26.0, [0.11775779049012938, 0.09127455921787042, 0.05967622327206553, 0.04917668786657249,
                    0.05281811495762224, 0.0312151700137975, 0.00917717106815441, 0.04911775537780748]),
        (17, 25.0, [0.10476763428036885, 0.08116376667294865, 0.05377037856847226, 0.04464020686233841,
                    0.046959164870810116, 0.029576209021535602, 0.01202141841728862, 0.04366776702312779,
                    0.011606418426675627]),
        (19, 24.0, [0.09276093045835816, 0.07182637494834421, 0.04809270537781395, 0.040086638135147665,
                    0.04154990918423161, 0.027536560854546758, 0.01415738625943351, 0.03863636785368814,
                    0.011602707907782908, 0.01184239437624057]),
    ]
    for sigma_max, theta0, expected in cases:
        result = random_csl_fractions(sigma_max, theta0)
        assert result.shape == (len(expected),)
        assert np.allclose(result, expected, rtol=1e-7, atol=1e-13)
    # the fraction of a class does not depend on classes of larger index
    assert np.allclose(random_csl_fractions(13, 25.0), random_csl_fractions(17, 25.0)[:7], rtol=1e-7, atol=1e-13)


## Solution

In [ ]:
import math

import numpy as np


def ball_overlap_measure(quats, radii):
    quats = np.atleast_2d(np.asarray(quats, dtype=float))
    quats = quats / np.linalg.norm(quats, axis=1)[:, None]
    half = 0.5 * np.radians(np.asarray(radii, dtype=float).reshape(-1))   # cap radii on the quaternion sphere S^3
    k = len(half)
    if np.any(half <= 0.0):
        return 0.0
    # integrate over the geodesic distance psi from the centre of the smallest cap
    order = np.argsort(half)
    quats, half = quats[order], half[order]
    p, alpha = quats[0], half[0]
    rules = []
    for n in (20, 40):
        nodes, weights = np.polynomial.legendre.leggauss(n)
        s_map = 0.5 * (nodes + 1.0)
        # smoothstep map: removes square-root behaviour at panel ends
        rules.append((s_map * s_map * (3.0 - 2.0 * s_map), 3.0 * s_map * (1.0 - s_map) * weights))

    def slice_geometry(q, beta, psi):
        # the slice {x on S^3 : angle(x, p) = psi} is a 2-sphere; its part inside cap(q, beta) is a spherical cap
        # (centred on the direction of q seen from p) with angular radius a(psi); returns cos(a) (a = 0: empty, pi: full)
        # and the angle d between p and q
        c = float(p @ q)
        s = float(np.linalg.norm(q - c * p))
        d = math.atan2(s, c)                                  # angle between p and q
        sp, cp = np.sin(psi), np.cos(psi)
        full = (psi + d <= beta) | (psi + d >= 2.0 * math.pi - beta)
        empty = np.abs(psi - d) >= beta
        with np.errstate(divide="ignore", invalid="ignore"):
            t = (math.cos(beta) - cp * math.cos(d)) / (sp * math.sin(d))
        t = np.where(full, -1.0, np.where(empty, 1.0, np.clip(t, -1.0, 1.0)))
        return t, d

    def lens_fraction(ca, cb, g):
        # fraction of the unit 2-sphere inside two caps (cosines of the radii ca, cb; centre separation g)
        a, b = np.arccos(ca), np.arccos(cb)
        sa, sb = np.sin(a), np.sin(b)
        out = np.zeros_like(a)
        small = np.minimum(a, b)
        contain = g <= np.abs(a - b)
        cover = a + b >= 2.0 * math.pi - g
        part = (g < a + b) & ~contain & ~cover & (sa > 0) & (sb > 0)
        out = np.where(contain, 0.5 * (1.0 - np.cos(small)), out)
        out = np.where(cover & ~contain, 0.5 * (1.0 - ca) + 0.5 * (1.0 - cb) - 1.0, out)
        if np.any(part) and math.sin(g) > 0.0:
            with np.errstate(divide="ignore", invalid="ignore"):
                x = np.clip((math.cos(g) - ca * cb) / (sa * sb), -1.0, 1.0)
                y = np.clip((cb - math.cos(g) * ca) / (math.sin(g) * sa), -1.0, 1.0)
                z = np.clip((ca - math.cos(g) * cb) / (math.sin(g) * sb), -1.0, 1.0)
                lens = 2.0 * (math.pi - np.arccos(x) - ca * np.arccos(y) - cb * np.arccos(z)) / (4.0 * math.pi)
            out = np.where(part, lens, out)
        out = np.where((a <= 0.0) | (b <= 0.0), 0.0, out)
        return out

    def area_fraction(others, psi):
        # fraction of the slice 2-sphere at distance psi from p that lies inside all other caps
        if not others:
            return np.ones_like(psi)
        if len(others) == 1:
            q, beta = others[0]
            t, _ = slice_geometry(q, beta, psi)
            return 0.5 * (1.0 - t)
        (q2, b2), (q3, b3) = others
        t2, _ = slice_geometry(q2, b2, psi)
        t3, _ = slice_geometry(q3, b3, psi)
        # angle between the directions of q2 and q3 as seen from p (tangent directions at p)
        u2 = q2 - float(p @ q2) * p
        u3 = q3 - float(p @ q3) * p
        n2, n3 = np.linalg.norm(u2), np.linalg.norm(u3)
        if n2 == 0.0 or n3 == 0.0:                            # a cap centred on +-p: it is either full or empty on the slice
            return np.where(t2 <= -1.0, 0.5 * (1.0 - t3), 0.0) if n2 == 0.0 else np.where(t3 <= -1.0, 0.5 * (1.0 - t2), 0.0)
        g = math.atan2(float(np.linalg.norm(u2 / n2 - u3 / n3)), float(np.linalg.norm(u2 / n2 + u3 / n3))) * 2.0
        return lens_fraction(t2, t3, g)

    def cap_intersection(others):
        # fraction of S^3 inside cap(p, alpha) and all other caps: (2/pi) int_0^alpha sin(psi)^2 A(psi) dpsi
        cuts = {0.0, alpha}
        for q, beta in others:
            d = math.atan2(float(np.linalg.norm(q - float(p @ q) * p)), float(p @ q))
            cuts.update(x for x in (beta - d, d - beta, d + beta, 2.0 * math.pi - beta - d) if 0.0 < x < alpha)
        if len(others) == 2:
            # regime changes of the two slice caps (touching, containment, covering) located by bisection
            grid = np.linspace(0.0, alpha, 801)[1:-1]
            (q2, b2), (q3, b3) = others
            u2 = q2 - float(p @ q2) * p
            u3 = q3 - float(p @ q3) * p
            if np.linalg.norm(u2) > 0.0 and np.linalg.norm(u3) > 0.0:
                g = 2.0 * math.atan2(float(np.linalg.norm(u2 / np.linalg.norm(u2) - u3 / np.linalg.norm(u3))),
                                     float(np.linalg.norm(u2 / np.linalg.norm(u2) + u3 / np.linalg.norm(u3))))

                def events(x):
                    s2, _ = slice_geometry(q2, b2, np.atleast_1d(x))
                    s3, _ = slice_geometry(q3, b3, np.atleast_1d(x))
                    a, b = np.arccos(s2), np.arccos(s3)
                    return np.stack([a + b - g, a - b - g, b - a - g, a + b - (2.0 * math.pi - g)], axis=-1)

                vals = events(grid)
                lo_list, hi_list, col_list = [], [], []
                for m in range(4):
                    sign = np.sign(vals[:, m])
                    for i in np.nonzero(sign[:-1] * sign[1:] < 0)[0]:
                        lo_list.append(grid[i])
                        hi_list.append(grid[i + 1])
                        col_list.append(m)
                if lo_list:
                    lo, hi, col = np.array(lo_list), np.array(hi_list), np.array(col_list)
                    rows = np.arange(len(col))
                    flo = events(lo)[rows, col]
                    for _ in range(60):                   # bisection of all brackets at once
                        mid = 0.5 * (lo + hi)
                        fm = events(mid)[rows, col]
                        same = (fm < 0) == (flo < 0)
                        lo = np.where(same, mid, lo)
                        flo = np.where(same, fm, flo)
                        hi = np.where(same, hi, mid)
                    cuts.update(float(x) for x in 0.5 * (lo + hi))
        cuts = sorted(cuts)

        def integrand(psi, lo, hi):
            if len(others) == 1:
                q, beta = others[0]
                d = math.atan2(float(np.linalg.norm(q - float(p @ q) * p)), float(p @ q))
                mid = 0.5 * (lo + hi)
                if abs(mid - d) >= beta:
                    return np.zeros_like(psi)
                if mid + d <= beta or mid + d >= 2.0 * math.pi - beta:
                    return np.sin(psi) ** 2
                # cancellation-free form of sin(psi)^2 (1 - t) / 2
                return np.sin(psi) * np.sin(0.5 * (beta + psi - d)) * np.sin(0.5 * (beta - psi + d)) / math.sin(d)
            return np.sin(psi) ** 2 * area_fraction(others, psi)

        def panel(lo, hi, depth):
            coarse, fine = [(hi - lo) * float(np.sum(w * integrand(lo + (hi - lo) * h, lo, hi))) for h, w in rules]
            if abs(fine - coarse) <= 1e-16 + 1e-12 * abs(fine) or depth >= 40:
                return fine
            mid = 0.5 * (lo + hi)
            return panel(lo, mid, depth + 1) + panel(mid, hi, depth + 1)

        total = 0.0
        for lo, hi in zip(cuts[:-1], cuts[1:]):
            if hi > lo:
                total += panel(lo, hi, 0)
        return 2.0 / math.pi * total

    # each ball corresponds to the two antipodal caps around +q and -q; the caps around +p and -p give equal shares
    total = 0.0
    signs = [()] if k == 1 else ([(1.0,), (-1.0,)] if k == 2 else [(1.0, 1.0), (1.0, -1.0), (-1.0, 1.0), (-1.0, -1.0)])
    for sg in signs:
        others = [(s * quats[j + 1], half[j + 1]) for j, s in enumerate(sg)]
        total += cap_intersection(others)
    return 2.0 * total


def cubic_disorientation_cdf(theta):
    theta = float(theta)
    if theta <= 0.0:
        return 0.0
    if theta >= 180.0:
        return 1.0

    # Fundamental zone of the 24 proper cubic rotations in Rodrigues space r = n tan(omega/2): a truncated cube bounded by
    # |r_i| <= tan(pi/8) (from the 90 degree rotations) and |r_1| + |r_2| + |r_3| <= 1 (from the 120 degree rotations).
    t8 = math.tan(math.pi / 8.0)
    normals, dists = [], []
    for k in range(3):
        for sgn in (1.0, -1.0):
            n = np.zeros(3)
            n[k] = sgn
            normals.append(n)
            dists.append(t8)
    for sx in (1.0, -1.0):
        for sy in (1.0, -1.0):
            for sz in (1.0, -1.0):
                normals.append(np.array([sx, sy, sz]) / math.sqrt(3.0))
                dists.append(1.0 / math.sqrt(3.0))
    normals = np.array(normals)
    dists = np.array(dists)
    cos_pairs = np.clip(normals @ normals.T, -1.0, 1.0)
    rho_edge = math.sqrt(2.0) * t8                              # nearest points of the edges
    rho_vertex = math.sqrt(2.0 * t8 * t8 + (1.0 - 2.0 * t8) ** 2)  # vertices (disorientation 62.80 degrees)

    def inside_fraction(rho):
        # fraction of the sphere |r| = rho inside the zone: caps beyond each face, corrected by the pairwise overlaps of
        # caps (edges); for rho below the vertex distance no point of the sphere lies beyond three faces
        if rho >= rho_vertex:
            return 0.0
        cap = np.arccos(np.clip(dists / rho, -1.0, 1.0))        # angular radii of the caps (0 if the face is not reached)
        f = 1.0 - np.sum(0.5 * (1.0 - np.cos(cap)))
        m = len(dists)
        for i in range(m):
            for j in range(i + 1, m):
                a, b = cap[i], cap[j]
                if a <= 0.0 or b <= 0.0:
                    continue
                c = math.acos(cos_pairs[i, j])
                if c >= a + b:
                    continue
                if c <= abs(a - b):
                    lens = 2.0 * math.pi * (1.0 - math.cos(min(a, b)))
                else:
                    x = (math.cos(c) - math.cos(a) * math.cos(b)) / (math.sin(a) * math.sin(b))
                    y = (math.cos(b) - math.cos(c) * math.cos(a)) / (math.sin(c) * math.sin(a))
                    z = (math.cos(a) - math.cos(c) * math.cos(b)) / (math.sin(c) * math.sin(b))
                    lens = 2.0 * (math.pi - math.acos(max(-1.0, min(1.0, x)))
                                  - math.cos(a) * math.acos(max(-1.0, min(1.0, y)))
                                  - math.cos(b) * math.acos(max(-1.0, min(1.0, z))))
                f += lens / (4.0 * math.pi)
        return f

    # Haar measure in Rodrigues space: d(mu) = dV / (pi^2 (1 + rho^2)^2); rotation angle <= theta  <=>  rho <= tan(theta/2)
    rho_max = min(math.tan(math.radians(theta) / 2.0) if theta < 180.0 else math.inf, rho_vertex)
    breaks = [0.0] + [x for x in (t8, 1.0 / math.sqrt(3.0), rho_edge) if x < rho_max] + [rho_max]
    nodes, weights = np.polynomial.legendre.leggauss(60)
    s = 0.5 * (nodes + 1.0)
    h = s * s * (3.0 - 2.0 * s)                                 # smoothstep map: removes endpoint singularities
    dh = 6.0 * s * (1.0 - s)
    total = 0.0
    for lo, hi in zip(breaks[:-1], breaks[1:]):
        rho = lo + (hi - lo) * h
        vals = np.array([inside_fraction(x) for x in rho]) * rho ** 2 / (1.0 + rho ** 2) ** 2
        total += 0.5 * (hi - lo) * float(np.sum(weights * vals * dh))
    return min(1.0, 24.0 * 4.0 / math.pi * total)


def random_csl_fractions(sigma_max, theta0=15.0):
    sigma_max = int(sigma_max)

    def sigma_rotations(sigma):
        # all proper rotations R with sigma the smallest positive integer making sigma * R integral (odd sigma only)
        if sigma == 1:
            r = np.arange(-1, 2)
        else:
            r = np.arange(-sigma, sigma + 1)
        x, y, z = np.meshgrid(r, r, r, indexing="ij")
        mask = x * x + y * y + z * z == sigma * sigma
        rows = np.stack([x[mask], y[mask], z[mask]], axis=1).astype(np.int64)
        i_idx, j_idx = np.nonzero(rows @ rows.T == 0)
        r1, r2 = rows[i_idx], rows[j_idx]
        cross = np.cross(r1, r2)
        ok = np.all(cross % sigma == 0, axis=1)
        mats = np.stack([r1[ok], r2[ok], cross[ok] // sigma], axis=1)
        common = np.gcd.reduce(mats.reshape(len(mats), 9), axis=1)
        mats = mats[np.gcd(common, sigma) == 1].astype(float) / sigma
        # rotation matrices -> unit quaternions (w, x, y, z), Shepperd's method (stable for all angles)
        quats = []
        for m in mats:
            tr = m[0, 0] + m[1, 1] + m[2, 2]
            k = int(np.argmax([tr, m[0, 0], m[1, 1], m[2, 2]]))
            if k == 0:
                s = 2.0 * math.sqrt(1.0 + tr)
                q = [0.25 * s, (m[2, 1] - m[1, 2]) / s, (m[0, 2] - m[2, 0]) / s, (m[1, 0] - m[0, 1]) / s]
            elif k == 1:
                s = 2.0 * math.sqrt(1.0 + m[0, 0] - m[1, 1] - m[2, 2])
                q = [(m[2, 1] - m[1, 2]) / s, 0.25 * s, (m[0, 1] + m[1, 0]) / s, (m[0, 2] + m[2, 0]) / s]
            elif k == 2:
                s = 2.0 * math.sqrt(1.0 - m[0, 0] + m[1, 1] - m[2, 2])
                q = [(m[0, 2] - m[2, 0]) / s, (m[0, 1] + m[1, 0]) / s, 0.25 * s, (m[1, 2] + m[2, 1]) / s]
            else:
                s = 2.0 * math.sqrt(1.0 - m[0, 0] - m[1, 1] + m[2, 2])
                q = [(m[1, 0] - m[0, 1]) / s, (m[0, 2] + m[2, 0]) / s, (m[1, 2] + m[2, 1]) / s, 0.25 * s]
            q = np.array(q)
            quats.append(q / np.linalg.norm(q))
        return np.array(quats)

    sigmas = list(range(1, sigma_max + 1, 2))
    centres, radii, labels = [], [], []
    for sigma in sigmas:
        q = sigma_rotations(sigma)
        if len(q) == 0:
            continue
        centres.append(q)
        radii.append(np.full(len(q), theta0 / math.sqrt(sigma)))
        labels.append(np.full(len(q), sigma))
    centres = np.concatenate(centres)
    radii = np.concatenate(radii)
    labels = np.concatenate(labels)

    fractions = np.zeros(len(sigmas))
    cache = {}
    for k, sigma in enumerate(sigmas):
        own_idx = np.nonzero(labels == sigma)[0]
        if len(own_idx) == 0:
            continue
        if sigma == 1:
            fractions[k] = cubic_disorientation_cdf(theta0)
            continue
        radius = theta0 / math.sqrt(sigma)
        value = len(own_idx) * ball_overlap_measure(centres[own_idx[:1]], [radius])
        # balls of index <= sigma that overlap a ball of index sigma
        cand = np.nonzero(labels <= sigma)[0]
        dots = np.abs(centres[own_idx] @ centres[cand].T)
        limit = np.cos(np.radians(radii[own_idx][:, None] + radii[cand][None, :]) / 2.0)
        neighbours = {}
        for a, b in zip(*np.nonzero(dots > limit)):
            i, j = int(own_idx[a]), int(cand[b])
            if i != j:
                neighbours.setdefault(i, []).append(j)

        def overlapping(i, j):
            return abs(float(centres[i] @ centres[j])) > math.cos(math.radians(radii[i] + radii[j]) / 2.0)

        def measure(idx):
            # the measure depends only on the radii and on the Gram matrix of the quaternions up to their signs;
            # symmetry-equivalent configurations recur many times, so results are cached
            g = centres[idx] @ centres[idx].T
            key = tuple(np.round(radii[idx], 12)) + tuple(np.round(np.abs(g[np.triu_indices(len(idx), 1)]), 12))
            if len(idx) == 3:
                key += (bool(g[0, 1] * g[0, 2] * g[1, 2] < 0.0),)
            if key not in cache:
                cache[key] = ball_overlap_measure(centres[idx], radii[idx])
            return cache[key]

        # inclusion-exclusion up to triple intersections (no rotation lies in four balls): the class receives
        # mu(A) - mu(A ∩ V), with A the union of its balls and V the union of the balls of smaller index
        for i, nbrs in neighbours.items():
            for j in nbrs:
                if labels[j] < sigma or j > i:        # each pair once: own-own pairs with j > i, all own-lower pairs
                    value -= measure([i, j])
            for x in range(len(nbrs)):
                for y in range(x + 1, len(nbrs)):
                    j, l = nbrs[x], nbrs[y]
                    # count a triple once, from its ball of index sigma with the smallest position
                    if (labels[j] == sigma and j < i) or (labels[l] == sigma and l < i) or not overlapping(j, l):
                        continue
                    value += measure([i, j, l])
        fractions[k] = value
    return fractions
